# Data Collection

**Project:** Developer Technology & Trends Survey Analysis — Data Analyst Capstone

This notebook covers the first stage of the project: sourcing and doing an initial structural check on the data before any cleaning happens. Two datasets are pulled in:

1. **Primary dataset** — a snapshot of the Stack Overflow Annual Developer Survey (65,457 responses × 114 columns), covering languages, databases, platforms, frameworks, AI-tool usage, demographics, and job satisfaction. This is the dataset the rest of the project — wrangling, EDA, visualization, and the Looker Studio dashboard — is built on.
2. **Supplementary dataset** — average annual salary by programming language, collected via web scraping. It's used later to check whether the languages people use most are also the ones that pay best.


In [1]:
import pandas as pd
import requests
from bs4 import BeautifulSoup

## 1. Primary dataset: Stack Overflow Developer Survey

The survey export is hosted as a CSV. Loading it straight from the URL keeps the notebook reproducible — no manual download step required.

In [3]:
survey_url = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/n01PQ9pSmiRX6520flujwQ/survey-data.csv"
df = pd.read_csv(survey_url)

KeyboardInterrupt: 

In [ ]:
# df = pd.read_csv('survey_data.csv')

### Shape and structure

In [ ]:
df.shape

65,457 responses across 114 columns — a wide survey, as expected from a multi-section developer questionnaire (background, tech stack, AI tools, compensation, satisfaction, etc.).

In [ ]:
df.dtypes.value_counts()

The large majority of columns are text/object — mostly categorical single-answer questions (e.g. `Country`, `EdLevel`) or semicolon-delimited multi-select questions (e.g. `LanguageHaveWorkedWith`). The 13 numeric columns are mostly compensation and job-satisfaction scores (`ConvertedCompYearly`, `JobSat`, `JobSatPoints_1`…`_11`).

### First look at the rows

In [ ]:
df.head()

A couple of things worth flagging for the next stage:

- **`Check`** is an attention-check column — genuine respondents should all have the value `Apples`. This is useful later for filtering out low-quality responses.
- Plenty of `NaN`s are expected and not a data-quality problem by themselves: the survey uses branching logic, so many questions simply don't apply to every respondent (e.g. `CodingActivities` is blank for students who haven't started a job yet).

### Quick demographic and geographic profile

In [ ]:
age_mapping = {
    "Under 18 years old": 17,
    "18-24 years old": 21,
    "25-34 years old": 29,
    "35-44 years old": 39,
    "45-54 years old": 49,
    "55-64 years old": 59,
    "65 years or older": 70,
}

df["Age_numeric"] = df["Age"].map(age_mapping)
df["Age_numeric"].mean()

Average respondent age works out to roughly **33 years old**, which lines up with the 25–34 age bracket being the largest single group in the dashboard's age breakdown.

In [ ]:
df["Country"].nunique()

In [ ]:
top_countries = df["Country"].value_counts().head(5).reset_index()
top_countries.columns = ["Country", "Count"]
top_countries

185 countries are represented. The United States accounts for the single largest share (~17% of all respondents on its own), with Germany, India, the UK, and Ukraine rounding out the top five — a genuinely global sample, but with a clear US skew to keep in mind when generalizing findings.

### A note on data quality

The source filename (`survey-data-with-duplicate.csv`) is a hint that this extract has duplicate rows mixed in on purpose. Confirming that, along with handling missing values and converting the multi-select columns into a more usable form, is covered in the next notebook (**Data Wrangling**) rather than here — this notebook's job is just to get the data in and confirm it looks like what we expect.

## 2. Supplementary dataset: programming language salary benchmarks (web scraping)

To add some context to the "most-used languages" findings later on, I scraped a small reference table of average annual salary by programming language. The source page has a simple HTML table, so `requests` + `BeautifulSoup` is all that's needed.

In [ ]:
salary_url = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DA0321EN-SkillsNetwork/labs/datasets/Programming_Languages.html"

page = requests.get(salary_url).text
soup = BeautifulSoup(page, "html.parser")
table = soup.find("table")

Parse each table row into a record, skipping the header row:

In [ ]:
records = []
for row in table.find_all("tr")[1:]:
    cols = row.find_all("td")
    records.append({
        "Language": cols[1].get_text(),
        "Avg Annual Salary (USD)": cols[3].get_text(),
    })

lang_salary_df = pd.DataFrame(records)
lang_salary_df

Only 10 languages, so this is more of a spot-check than a full benchmark — but a useful one. **Swift** and **Python** top the list, while **PHP**, **C#**, and **SQL** sit at the bottom despite SQL being one of the most *used* languages in the main survey. That mismatch between "widely used" and "well paid" is worth revisiting once the main survey data is cleaned and charted.

### Save both datasets

In [ ]:
import os

os.makedirs("./data/raw", exist_ok=True)

In [ ]:
df.to_csv("./data/raw/survey_data_raw.csv", index=False)
lang_salary_df.to_csv("./data/raw/language_salaries.csv", index=False)

## Summary & next steps

Collected:
- **Primary survey data**: 65,457 responses × 114 columns, loaded and structurally verified.
- **Supplementary data**: 10-language salary benchmark, scraped and saved.

Flagged for the next stage:
- Likely duplicate rows in the survey extract (filename suggests this is intentional)
- Missing values driven by survey branching logic (expected, not necessarily a problem)
- Multi-select columns (`LanguageHaveWorkedWith`, `DatabaseHaveWorkedWith`, etc.) stored as delimited strings, which will need splitting before they're usable for aggregation and charting

→ Continues in **02_Data_Wrangling.ipynb**.